# Process Discovery Lab
### From Event Log Exploration to Discovery Optimization with PM4Py

**Learning goals**
- Explore an event log and its variant distribution.
- Compare two miners (Inductive Miner and Alpha Miner) on the event log.
- Apply top-k and coverage-based variant filtering.
- Implement iterative variant selection.
- Tune Inductive Miner Infrequent.
- Compare model quality, complexity and discovery time.

> **Note.** Run the notebook from top to bottom (*Runtime → Restart and run all*). Every function is defined once, and all filters share the same definition of *variant* (the sequence of activities of a case, ordered by timestamp).

## 0. Setup

In [ ]:
# If needed, uncomment and run:
# !pip install pm4py pandas matplotlib numpy

import random
import time
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

import pm4py

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CASE_KEY = "case:concept:name"
ACT_KEY = "concept:name"
TS_KEY = "time:timestamp"

print("PM4Py version:", getattr(pm4py, "__version__", "unknown"))

## 1. Import event logs

In [ ]:
LOG_PATH = "sp2020.xes"   # adjust to your environment (e.g. "/content/sp2020.xes" on Colab)

bpic20r = pm4py.read_xes(LOG_PATH)

logs = {"bpic20r": bpic20r}

## 2. Log exploration

In [ ]:
def case_variants(log):
    # One tuple of activities per case, events ordered by timestamp (stable sort).
    # This is the single definition of 'variant' used throughout the notebook.
    ordered = log
    if TS_KEY in ordered.columns:
        ordered = ordered.sort_values(TS_KEY, kind="stable")
    return ordered.groupby(CASE_KEY, sort=False)[ACT_KEY].agg(tuple)

def variant_counts(log):
    # Variants ranked by frequency (index = tuple of activities, value = number of cases).
    return case_variants(log).value_counts()

def log_summary(log, name="log"):
    lengths = log.groupby(CASE_KEY).size()
    return {
        "log": name,
        "cases": log[CASE_KEY].nunique(),
        "events": len(log),
        "activities": log[ACT_KEY].nunique(),
        "variants": len(variant_counts(log)),
        "avg_trace_length": lengths.mean(),
        "min_trace_length": lengths.min(),
        "max_trace_length": lengths.max(),
    }

summary = pd.DataFrame([log_summary(log, name) for name, log in logs.items()])
summary

In [ ]:
def plot_variant_distribution(log, title, top=20):
    counts = variant_counts(log).head(top)
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar([f"V{i}" for i in range(1, len(counts) + 1)], counts.values)
    ax.set_title(f"Top {top} variants — {title}  (V1 = most frequent)")
    ax.set_xlabel("Variant rank")
    ax.set_ylabel("Case count")
    plt.tight_layout()
    plt.show()

def plot_variant_coverage(log, title):
    counts = variant_counts(log)
    cumulative = counts.cumsum() / counts.sum()
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(range(1, len(cumulative) + 1), cumulative.values, marker=".")
    ax.axhline(0.80, linestyle="--", label="80% coverage")
    ax.set_title(f"Cumulative variant coverage — {title}")
    ax.set_xlabel("Variants included (ranked by frequency)")
    ax.set_ylabel("Case coverage")
    ax.legend()
    plt.tight_layout()
    plt.show()

for name, log in logs.items():
    plot_variant_distribution(log, name)
    plot_variant_coverage(log, name)

In [ ]:
def variants_needed_for_coverage(log, targets=(0.80, 0.90, 0.95)):
    counts = variant_counts(log)
    cumulative = counts.cumsum() / counts.sum()
    return {t: int(np.argmax(cumulative.values >= t - 1e-12) + 1) for t in targets}

coverage_table = pd.DataFrame({
    name: variants_needed_for_coverage(log) for name, log in logs.items()
}).T
coverage_table

## 3. Preprocessing — Top-k variants

We compare several values of `k`. To stay consistent with the rest of the notebook, top-k filtering is implemented on top of the same `case_variants` helper (equivalent to `pm4py.filter_variants_top_k`, up to tie-breaking between equally frequent variants).

In [ ]:
def filter_by_variants(log, selected_variants):
    # Keep only the cases whose variant is in `selected_variants` (a set of tuples).
    cv = case_variants(log)
    keep = cv.index[cv.apply(selected_variants.__contains__)]
    return log[log[CASE_KEY].isin(keep)].copy()

def select_top_k(log, k):
    return set(variant_counts(log).index[:k])

def filter_top_k(log, k):
    return filter_by_variants(log, select_top_k(log, k))

def retained_case_percentage(original, filtered):
    return 100 * filtered[CASE_KEY].nunique() / original[CASE_KEY].nunique()

def retained_event_percentage(original, filtered):
    return 100 * len(filtered) / len(original)

def retained_k_variants(log, k_values=(1, 2, 3, 5, 10, 20)):
    rows = []
    for k in k_values:
        filtered = filter_top_k(log, k)
        rows.append({
            "k": k,
            "cases_retained_%": retained_case_percentage(log, filtered),
            "events_retained_%": retained_event_percentage(log, filtered),
            "variants_retained": len(variant_counts(filtered)),
        })
    return pd.DataFrame(rows)

topk_bpic20r = retained_k_variants(bpic20r)
display(topk_bpic20r)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(topk_bpic20r["k"], topk_bpic20r["cases_retained_%"], marker="o")
ax.set_xlabel("Top-k variants")
ax.set_ylabel("Cases retained (%)")
ax.set_title("Top-k variant filtering — retained case coverage")
plt.tight_layout()
plt.show()

## 4. Preprocessing — Coverage-based filtering

Instead of choosing a number of variants, choose the percentage of cases that should be covered by the selected variants: variants are taken in decreasing frequency until the target coverage is reached.

In [ ]:
def select_variants_by_coverage(log, target):
    counts = variant_counts(log)
    cumulative = counts.cumsum() / counts.sum()
    n = int(np.argmax(cumulative.values >= target - 1e-12)) + 1
    return set(counts.index[:n])

def filter_variants_by_cumulative_coverage(log, target):
    if CASE_KEY not in log.columns or ACT_KEY not in log.columns:
        raise ValueError(f"Missing columns: need {CASE_KEY} and {ACT_KEY}")
    return filter_by_variants(log, select_variants_by_coverage(log, target))

def coverage_report(log, targets=(0.70, 0.80, 0.90, 0.95, 0.99)):
    rows = []
    for target in targets:
        selected = select_variants_by_coverage(log, target)
        filtered = filter_by_variants(log, selected)
        rows.append({
            "target_coverage": target,
            "cases_retained_%": round(retained_case_percentage(log, filtered), 2),
            "events_retained_%": round(retained_event_percentage(log, filtered), 2),
            "variants_retained": len(selected),
        })
    return pd.DataFrame(rows)

In [ ]:
coverage_bpic20r = coverage_report(bpic20r)
print("bpic20r")
display(coverage_bpic20r)

## 5. Discovery algorithms

We create wrappers for a common set of discovery algorithms. Discovery APIs may vary slightly across PM4Py releases; this section uses the high-level PM4Py functions.

In [ ]:
def discover_inductive_petri_net(log, noise_threshold=0.0):
    # noise_threshold = 0.0 -> Inductive Miner; > 0 -> Inductive Miner Infrequent (IMf)
    return pm4py.discover_petri_net_inductive(log, noise_threshold=noise_threshold)

def discover_alpha_net(log):
    return pm4py.discover_petri_net_alpha(log)

def visualize_petri_net(net, initial_marking, final_marking):
    pm4py.view_petri_net(net, initial_marking, final_marking)

def timed(fn, *args, **kwargs):
    # Returns (result, elapsed seconds)
    t0 = time.perf_counter()
    result = fn(*args, **kwargs)
    return result, time.perf_counter() - t0

### 5.1 Inductive Miner

Discover and visualize a Petri net on the (unfiltered) log with `noise_threshold=0.0`.

In [ ]:
im_net, im_initial_marking, im_final_marking = discover_inductive_petri_net(bpic20r, noise_threshold=0.0)
visualize_petri_net(im_net, im_initial_marking, im_final_marking)

### 5.2 Alpha Miner

In [ ]:
alpha_net, alpha_im, alpha_fm = discover_alpha_net(bpic20r)
visualize_petri_net(alpha_net, alpha_im, alpha_fm)

## 6. Model complexity indicators

We use basic structural indicators as a simple complexity proxy. These are not a complete measure of model complexity or interpretability.

In [ ]:
def petri_net_size(net):
    return {
        "places": len(net.places),
        "transitions": len(net.transitions),
        "arcs": len(net.arcs),
        "visible_transitions": sum(1 for t in net.transitions if t.label is not None),
        "silent_transitions": sum(1 for t in net.transitions if t.label is None),
    }

print("Alpha:", petri_net_size(alpha_net))
print("IM   :", petri_net_size(im_net))

## 7. Inductive Miner Infrequent — threshold sweep

We use the Inductive Miner implementation with its `noise_threshold` parameter. The values below are exploratory: the useful range depends on the log. Start with a modest sweep and expand it if needed.

The sweep only reports **structural** size. Model size is not model quality, and the relation between threshold and size need not be monotonic (IMf filtering is heuristic, and a plateau or an increase in visible transitions is possible); quality is assessed in sections 8-9.

In [ ]:
def imf_sweep(log, thresholds=(0.0, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40)):
    models, rows = {}, []
    for threshold in thresholds:
        net, im, fm = discover_inductive_petri_net(log, noise_threshold=threshold)
        models[threshold] = (net, im, fm)
        rows.append({"threshold": threshold, **petri_net_size(net)})
    return pd.DataFrame(rows), models

bpic20r_imf_results, bpic20r_imf_models = imf_sweep(bpic20r)
display(bpic20r_imf_results)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(bpic20r_imf_results["threshold"], bpic20r_imf_results["transitions"], marker="o")
ax.set_xlabel("Noise threshold")
ax.set_ylabel("Number of transitions")
ax.set_title("IMf threshold vs model size")
plt.tight_layout()
plt.show()

## 8. Evaluation — fitness and precision

The evaluation functions can differ across PM4Py versions and depend on the model type. This section provides a small wrapper using alignments replay where supported. If a metric is unavailable in your installed version, inspect the PM4Py evaluation API and adapt the call.

In [ ]:
def _extract_fitness(fit):
    if isinstance(fit, dict):
        for key in ("averageFitness", "average_trace_fitness", "log_fitness"):
            if key in fit:
                return float(fit[key])
        raise KeyError(f"No known fitness key in {list(fit)}")
    return float(fit)

def evaluate_petri_net(log, net, im, fm):
    # Alignment-based fitness, precision and their harmonic mean (F-measure) of `net` on `log`.
    metrics = {}
    fit = prec = None
    try:
        fit = _extract_fitness(pm4py.fitness_alignments(log, net, im, fm))
        metrics["fitness"] = fit
    except Exception as exc:
        metrics["fitness_error"] = str(exc)

    try:
        prec = float(pm4py.precision_alignments(log, net, im, fm))
        metrics["precision"] = prec
    except Exception as exc:
        metrics["precision_error"] = str(exc)

    if fit is not None and prec is not None:
        metrics["fmeasure"] = 2 * prec * fit / (prec + fit) if (prec + fit) > 0 else 0.0
    else:
        metrics["fmeasure_error"] = "fitness or precision unavailable"
    return metrics

# Example: evaluate one IMf model against the original log
net, im, fm = discover_inductive_petri_net(bpic20r, noise_threshold=0.10)
print(evaluate_petri_net(bpic20r, net, im, fm))
print(petri_net_size(net))

**Important methodological note:** evaluate filtered models against both (a) the filtered training log and (b) the original log. Otherwise, preprocessing can appear to improve quality simply because the evaluation log has also been simplified. In particular, a model discovered with `noise_threshold=0.0` always replays its own training log perfectly (fitness = 1.0), so metrics on the filtered log alone say very little. The experiment runner below always reports both.

## 9. Experiment runner

A compact runner makes it possible to compare preprocessing strategies systematically. For each strategy it records the time spent on sampling and on discovery, and the metrics on both the **training log** (`*_train`) and the **original log** (`*_orig`).

In [ ]:
def run_experiment(original_log, train_log, name, noise_threshold=0.0, sampling_time=0.0):
    (net, im, fm), discovery_time = timed(
        discover_inductive_petri_net, train_log, noise_threshold=noise_threshold
    )
    m_orig = evaluate_petri_net(original_log, net, im, fm)
    m_train = m_orig if train_log is original_log else evaluate_petri_net(train_log, net, im, fm)

    row = {
        "experiment": name,
        "cases": train_log[CASE_KEY].nunique(),
        "events": len(train_log),
        "variants": len(variant_counts(train_log)),
        "noise_threshold": noise_threshold,
        "sampling_time_s": sampling_time,
        "discovery_time_s": discovery_time,
    }
    row.update({f"{k}_train": v for k, v in m_train.items()})
    row.update({f"{k}_orig": v for k, v in m_orig.items()})
    row.update(petri_net_size(net))
    return row

experiment_rows = [run_experiment(bpic20r, bpic20r, "Original", 0.0)]

for k in [3, 5, 10]:
    filtered, t_sampling = timed(filter_top_k, bpic20r, k)
    experiment_rows.append(run_experiment(bpic20r, filtered, f"Top-{k}", 0.0, t_sampling))

for target in [0.80, 0.90, 0.95]:
    filtered, t_sampling = timed(filter_variants_by_cumulative_coverage, bpic20r, target)
    experiment_rows.append(run_experiment(bpic20r, filtered, f"Coverage-{target:.0%}", 0.0, t_sampling))

experiment_df = pd.DataFrame(experiment_rows)
experiment_df

## 10. Visual comparison of experiment results

Inspect the resulting metrics and discuss the model size versus quality trade-off. Be careful not to treat model size as a direct measure of model quality. The left panel (original log) is the fair comparison; the right panel (training log) is shown to illustrate how misleading it is.

In [ ]:
metric_cols = ["fitness", "precision", "fmeasure"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, suffix, title in zip(axes, ["orig", "train"],
                             ["Evaluated on the ORIGINAL log", "Evaluated on the TRAINING (filtered) log"]):
    cols = [f"{m}_{suffix}" for m in metric_cols if f"{m}_{suffix}" in experiment_df.columns]
    experiment_df.set_index("experiment")[cols].plot(kind="bar", ax=ax)
    ax.set_title(title)
    ax.set_ylabel("Metric value")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 11. Discovery time improvement

Sampling is only worthwhile if it pays off in time. Following the usual definitions:

$$\text{DiscoveryTime}_{Improvement} = \frac{\text{DiscoveryTime}_{WholeLog}}{\text{DiscoveryTime}_{SampledLog}} \qquad (5)$$

$$\text{TotalTime}_{Improvement} = \frac{\text{DiscoveryTime}_{WholeLog}}{\text{DiscoveryTime}_{SampledLog} + \text{SamplingTime}} \qquad (6)$$

Values greater than 1 mean the sampled approach is faster than discovering on the whole log.

In [ ]:
def add_time_improvements(df, baseline="Original"):
    df = df.copy()
    whole = df.loc[df["experiment"] == baseline, "discovery_time_s"].iloc[0]
    eps = 1e-9
    df["discovery_time_improvement"] = whole / (df["discovery_time_s"] + eps)
    df["total_time_improvement"] = whole / (df["discovery_time_s"] + df["sampling_time_s"] + eps)
    return df

time_df = add_time_improvements(experiment_df)
time_df[["experiment", "sampling_time_s", "discovery_time_s",
         "discovery_time_improvement", "total_time_improvement",
         "fmeasure_orig"]]

## 12. Final challenge

### Task
* Implement an iterative strategy based on F-measure improvements with a **frequency-based variants** approach.
* Implement an iterative strategy based on F-measure improvements with a **filter-based** approach (Inductive Miner noise threshold).

In both strategies the candidate model is evaluated against the **original log**, and a candidate is accepted only if it improves the best F-measure so far. Each procedure records its own total time, so it can be fed into the time-improvement formulas of section 11.